In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from data import UncertainDataset, ds_single, ds_pressure

In [3]:
import fcbench

In [4]:
from tqdm import tqdm

In [5]:
def compute_absolute_error_bound(v: xr.DataArray, time_chunk: int = 25) -> tuple[float, float]:
    vrange = float(np.nanmax(v) - np.nanmin(v))

    if vrange == 0.0:
        return (0.0, 1.0)

    prec = int(np.ceil(np.log10(vrange)))
    all_us = [0.0]
    
    while True:
        us = []
        
        for i in range(0, v.valid_time.size, time_chunk):
            vi = v.isel(valid_time=slice(i, i+time_chunk)).compute()
            
            v_linquant = np.round(vi / (10**prec)) * (10**prec)
            v_err = v_linquant - vi
            
            us.append(fcbench.metrics.Uniformity.goodness(v_err))
    
            break

        all_us.append(np.amin(us))

        if len(all_us) >= 5 and all(p <= all_us[-5] for p in all_us[-5:]):
            return 10**(prec + 4), all_us[-5]

        prec -= 1

In [6]:
with open("uniformity-single.txt", "w") as f:
    for key in tqdm(ds_single.mean):
        print(f"{key}: ", end="", flush=True, file=f)
        print(f"{compute_absolute_error_bound(ds_single.mean[key])!r}", flush=True, file=f)

100%|██████████| 226/226 [49:19<00:00, 13.10s/it]


In [7]:
with open("uniformity-pressure.txt", "w") as f:
    for key in tqdm(ds_pressure.mean, position=0):
        print(f"{key}:", flush=True, file=f)
        for pressure in tqdm(ds_pressure.mean[key].pressure_level.values, position=1):
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(f"{compute_absolute_error_bound(ds_pressure.mean[key].sel(pressure_level=pressure))!r}", flush=True, file=f)

100%|██████████| 16/16 [17:07<00:00, 64.23s/it]
